In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#data classes
import xarray as xr

#loading bar
from tqdm import tqdm

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallData"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    elif (RunType[0] == "TRACER") and (RunType[1] == "DRY"):
        SimulationTime = ("2022-06-08","2022-06-11")
    return SimulationTime

spinup_hours = 24
# spinup_hours = 12

# RunType = ("TRACER","MOIST","NSSL",spinup_hours)
RunType = ("TRACER","DRY","NSSL",spinup_hours)
SimulationTime = GetSimulationTime(RunType)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
#DATA INFORMATION
# NOAA nClimGrid-Daily Version 1 – Daily gridded temperature and precipitation for the Contiguous United States since 1951
# https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C01589

# The product referred to as nClimGrid-Daily is a set of daily gridded fields and area averages of temperature and precipitation
# that covers the Contiguous United States (CONUS) from 1951 to present and is updated daily. It is related to the monthly version
# of NClimGrid and NClimDiv, but with a daily temporal resolution. The gridded fields are stored in netCDF format with one file per data month.
# Area averages for nine types of regions are provided in CSV format with one file per region type and data month. 
# At a resolution of approximately 0.0417 degrees latitude and longitude (nominally 5-km grid), 
# the gridded data provide smoothed representations of the point observations. 
# Since the accuracy of estimates for individual grid points and days can be sensitive to local spatial variability 
# and the ability of the available observations and interpolation technique to capture that variability, 
# the nClimGrid-Daily dataset is recommended for applications that require the aggregation of estimates in space and/or time,
# such as climate monitoring analyses at regional to national scales.

In [ ]:
########################
#DATA RETRIEVAL FUNCTIONS

In [ ]:
#IMPORT FUNCTION LIBRARIES
import requests

In [ ]:
def DownloadPrecipData(yearmonth, outputDirectory):
    # Ensure the directory exists
    os.makedirs(outputDirectory, exist_ok=True)

    year = yearmonth[:4]

    fileUrl = (
        f"https://www.ncei.noaa.gov/thredds/fileServer/"
        f"nclimgrid-daily/{year}/ncdd-{yearmonth}-grd-scaled.nc"
    )

    # Build full output path
    outputPath = os.path.join(
        outputDirectory,
        f"ncdd-{yearmonth}-grd-scaled.nc"
    )

    response = requests.get(fileUrl, stream=True)
    response.raise_for_status()

    with open(outputPath, "wb") as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)

    print("Downloaded:", outputPath)


In [ ]:
#########################
#RUNNING

yearmonths = sorted({datestring.replace('-','')[:6] for datestring in ModelData.simulationDates[:-1]})
outputDirectory = os.path.join(DirectoryManager.dataDirectory,
                          f"Observation_Data/{ModelData.region}/nClimGrid_PrecipData")
for yearmonth in yearmonths:
    DownloadPrecipData(yearmonth,outputDirectory)